In [ ]:
import os
import gc
import argparse
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error, mean_absolute_error

# =============================
# Paths
# =============================
MOVIES_PATH = r"movies.csv"
RATINGS_PATH = r"ratings.csv"

# =============================
# I/O helpers
# =============================

def load_data(movies_path, ratings_path):
    """Load CSVs and normalize column names."""
    if not os.path.exists(movies_path) or not os.path.exists(ratings_path):
        raise FileNotFoundError("One or both input files do not exist.")

    movies_df = pd.read_csv(movies_path)
    movies_df = movies_df.rename(columns={"MovieID": "movieId", "Title": "title", "Genres": "genres"})

    ratings_df = pd.read_csv(ratings_path)
    ratings_df = ratings_df.rename(columns={"UserID": "userId", "MovieID": "movieId", "Rating": "rating"})

    for df in (movies_df, ratings_df):
        if "movieId" in df.columns:
            df["movieId"] = pd.to_numeric(df["movieId"], errors="ignore")
        if "userId" in df.columns:
            df["userId"] = pd.to_numeric(df["userId"], errors="ignore")
    return movies_df, ratings_df

# =============================
# CF utilities
# =============================

def _row_topk_prune(S, k):
    """Keep top-k positive entries per row; others -> 0. Assumes diagonal already 0."""
    if k is None or k <= 0:
        return np.zeros_like(S)
    n = S.shape[0]
    if k >= n:
        return np.where(S > 0, S, 0.0)
    out = np.zeros_like(S)
    for i in range(n):
        row = S[i]
        if np.count_nonzero(row) == 0:
            continue
        idx = np.argpartition(row, -k)[-k:]
        mask = row[idx] > 0
        if np.any(mask):
            out[i, idx[mask]] = row[idx[mask]]
    return out


def create_utility_matrix(ratings_df, train_movie_ids, k=2, like_threshold=3.5):
    """Return binary, normalized, similarity, completed_norm, completed_util, user_means."""
    filtered = ratings_df[ratings_df["movieId"].isin(train_movie_ids)]
    utility = filtered.pivot(index="userId", columns="movieId", values="rating")

    binary = (utility >= like_threshold).astype(int)

    user_means = utility.mean(axis=1, skipna=True)
    normalized = utility.sub(user_means, axis=0)

    norm_filled = normalized.fillna(0.0)
    sim = cosine_similarity(norm_filled)
    np.fill_diagonal(sim, 0.0)
    sim = np.where(sim > 0, sim, 0.0)

    S_topk = _row_topk_prune(sim, k=k)

    R = norm_filled.values
    I = (~normalized.isna()).astype(float).values
    num = S_topk @ R
    den = S_topk @ I
    with np.errstate(divide='ignore', invalid='ignore'):
        completed_norm = np.divide(num, den, out=np.zeros_like(num), where=den > 0)

    completed_norm_df = pd.DataFrame(completed_norm, index=normalized.index, columns=normalized.columns)
    obs_mask = normalized.notna()
    completed_norm_df[obs_mask] = normalized[obs_mask]

    completed_util_df = completed_norm_df.add(user_means, axis=0)
    util_obs_mask = utility.notna()
    completed_util_df[util_obs_mask] = utility[util_obs_mask]

    similarity_df = pd.DataFrame(S_topk, index=normalized.index, columns=normalized.index)

    return binary, normalized, similarity_df, completed_norm_df, completed_util_df, user_means


def get_random_movies_from_other_users(ratings_df, movies_df, user_id, num_movies=5, rng=None):
    if rng is None:
        rng = np.random.default_rng(42)
    user_movie_ids = ratings_df[ratings_df["userId"] == user_id]["movieId"].unique()
    other_movies = ratings_df[~ratings_df["movieId"].isin(user_movie_ids)]["movieId"].unique()
    if len(other_movies) == 0:
        return pd.DataFrame(columns=["userId", "movieId", "title", "genres", "rating"])
    selected = rng.choice(other_movies, size=min(num_movies, len(other_movies)), replace=False)
    out = movies_df[movies_df["movieId"].isin(selected)][["movieId", "title", "genres"]].copy()
    out["userId"] = user_id
    out["rating"] = pd.NA
    return out


def predict_rating(user_id, movie_id, similarity_matrix, ratings_df, user_means, k=2, min_rating=0.5, max_rating=5.0):
    """KNN prediction with top-k neighbors who've rated the movie; exclude self; clip to range."""
    if user_id not in similarity_matrix.index:
        return float(np.clip(user_means.get(user_id, 0.0), min_rating, max_rating))

    rated_users = ratings_df[ratings_df["movieId"] == movie_id]["userId"].unique()
    rated_users = [u for u in rated_users if u != user_id and u in similarity_matrix.index]
    if len(rated_users) == 0:
        return float(np.clip(user_means.get(user_id, 0.0), min_rating, max_rating))

    sims = similarity_matrix.loc[user_id, rated_users]
    if sims.empty:
        return float(np.clip(user_means.get(user_id, 0.0), min_rating, max_rating))

    k_eff = min(k, len(sims))
    top_k = sims.nlargest(k_eff)

    w_sum = 0.0
    s_sum = 0.0
    for v, s in top_k.items():
        r_vi = ratings_df[(ratings_df["userId"] == v) & (ratings_df["movieId"] == movie_id)]["rating"].values[0]
        mu_v = user_means.get(v, 0.0)
        w_sum += s * (r_vi - mu_v)
        s_sum += s
    pred = user_means.get(user_id, 0.0) + (w_sum / s_sum) if s_sum > 0 else user_means.get(user_id, 0.0)
    return float(np.clip(pred, min_rating, max_rating))

# =============================
# Evaluation
# =============================

def evaluate_predictions_console(df, user_id):
    """Print RMSE/MAE on rows with ground-truth only (optional)."""
    if df is None or df.empty:
        print(f"[User {user_id}] No data to evaluate.")
        return
    mask = df["rating"].notna()
    if not mask.any():
        print(f"[User {user_id}] No ground-truth ratings in test to evaluate.")
        return
    y_true = pd.to_numeric(df.loc[mask, "rating"], errors="coerce")
    y_pred = pd.to_numeric(df.loc[mask, "predicted_rating"], errors="coerce")
    valid = y_true.notna() & y_pred.notna()
    if not valid.any():
        print(f"[User {user_id}] No valid rows for evaluation.")
        return
    rmse = float(np.sqrt(mean_squared_error(y_true.loc[valid], y_pred.loc[valid])))
    mae = float(mean_absolute_error(y_true.loc[valid], y_pred.loc[valid]))
    print(f"[User {user_id}] RMSE={rmse:.4f} | MAE={mae:.4f}")


def precision_recall_at_k(df, like_threshold, k):
    """Compute (precision@K, recall@K) using only rows with ground-truth.
    Relevant := rating >= threshold; TopK := top k by predicted_rating.
    Precision@K divides by K even if |test| < K (as requested).
    """
    sub = df[df["rating"].notna()].copy()
    if sub.empty:
        return (np.nan, np.nan)
    sub["is_rel"] = sub["rating"] >= like_threshold
    sub = sub.sort_values("predicted_rating", ascending=False)
    topk = sub.head(int(k))
    hits = int(topk["is_rel"].sum())
    rel_total = int(sub["is_rel"].sum())
    precision = hits / k if k > 0 else np.nan
    recall = hits / rel_total if rel_total > 0 else np.nan
    return float(precision), (float(recall) if rel_total > 0 else np.nan)

# =============================
# Resume & averaging helpers
# =============================

def _processed_list_path(eval_root):
    return os.path.join(eval_root, "_processed_users.txt")


def load_processed_users(eval_root="evaluate"):
    """Read the checkpoint list of processed userIds (one per line)."""
    path = _processed_list_path(eval_root)
    if not os.path.exists(path):
        return set()
    out = set()
    try:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    out.add(int(line))
                except ValueError:
                    out.add(line)  # fallback if not int
    except Exception:
        pass
    return out


def append_processed_user(user_id, eval_root="evaluate"):
    """Append a processed userId to the checkpoint file."""
    os.makedirs(eval_root, exist_ok=True)
    path = _processed_list_path(eval_root)
    try:
        with open(path, "a", encoding="utf-8") as f:
            f.write(f"{int(user_id)}\n")
    except Exception:
        pass


def recompute_global_avg(eval_root="evaluate"):
    """Recompute avg_precision/avg_recall from all per-user files on disk."""
    if not os.path.exists(eval_root):
        print("[AVG] No evaluate dir found.")
        return
    rows = []
    for name in os.listdir(eval_root):
        if not name.startswith("user_"):
            continue
        user_dir = os.path.join(eval_root, name)
        if not os.path.isdir(user_dir):
            continue
        for fname in os.listdir(user_dir):
            if fname.startswith("evaluate_") and fname.endswith(".csv"):
                fpath = os.path.join(user_dir, fname)
                try:
                    d = pd.read_csv(fpath)
                    if {"precision", "recall"}.issubset(d.columns) and len(d) > 0:
                        rows.append(d[["precision", "recall"]].iloc[0])
                except Exception:
                    pass
    if len(rows) == 0:
        print("[AVG] No user metrics collected on disk.")
        return
    df_all = pd.DataFrame(rows)
    avg_p = float(np.nanmean(df_all["precision"])) if df_all["precision"].notna().any() else float("nan")
    avg_r = float(np.nanmean(df_all["recall"])) if df_all["recall"].notna().any() else float("nan")
    os.makedirs(eval_root, exist_ok=True)
    pd.DataFrame([{ "avg_precision": avg_p, "avg_recall": avg_r }]).to_csv(
        os.path.join(eval_root, "avg_evaluate.csv"), index=False
    )
    print(f"[AVG (disk)] precision={avg_p:.4f} | recall={avg_r:.4f}")

# =============================
# Main
# =============================

def main(k_neighbors=2, like_threshold=3.5, seed=42, k_at=10,
         start_at=0, limit=300, resume=False, force=False,
         eval_root="evaluate"):

    movies_df, ratings_df = load_data(MOVIES_PATH, RATINGS_PATH)

    try:
        min_rating = float(ratings_df["rating"].min())
        max_rating = float(ratings_df["rating"].max())
    except Exception:
        min_rating, max_rating = 0.5, 5.0

    rng = np.random.default_rng(seed)

    merged_df = ratings_df.merge(movies_df, on="movieId", how="inner")
    user_ids = ratings_df["userId"].unique()

    # ===== Resume/Batch planning =====
    orig_user_ids = sorted(list(user_ids))  # stable order across runs
    processed_set = load_processed_users(eval_root=eval_root)

    def has_eval(uid):
        return os.path.exists(os.path.join(eval_root, f"user_{uid}", f"evaluate_{uid}.csv"))

    if resume and not force:
        planned = [uid for uid in orig_user_ids if (uid not in processed_set) and (not has_eval(uid))]
    else:
        planned = orig_user_ids

    # Slice window
    if start_at > 0 or (limit and limit > 0):
        start = max(0, int(start_at))
        end = start + int(limit) if limit and limit > 0 else None
        planned = planned[start:end]
    print(f"[INFO] Planning to process {len(planned)} users in this run. (resume={'ON' if resume and not force else 'OFF'})")

    for user_id in tqdm(planned, desc="Processing users"):
        user_movies = merged_df[merged_df["userId"] == user_id]
        if user_movies.empty:
            print(f"Skipping user {user_id}: No data available.")
            continue

        try:
            train_data, test_data = train_test_split(user_movies, test_size=0.2, random_state=seed)
        except ValueError:
            print(f"Skipping user {user_id}: Not enough data to split.")
            continue
        if len(train_data) == 0 or len(test_data) == 0:
            print(f"Skipping user {user_id}: Not enough data to split.")
            continue

        train_movie_ids = train_data["movieId"].unique()

        (binary, normalized, similarity, completed_norm, completed_util, user_means) = create_utility_matrix(
            ratings_df, train_movie_ids, k=k_neighbors, like_threshold=like_threshold
        )

        # Dump matrices (optional)
        matrix_dir = os.path.join("Ma_tran_ban_dau", f"user_{user_id}")
        try:
            os.makedirs(matrix_dir, exist_ok=True)
            binary.to_csv(os.path.join(matrix_dir, f"ma_tran_ban_dau_{user_id}.csv"))
            normalized.to_csv(os.path.join(matrix_dir, f"ma_tran_tien_ich_chuan_hoa_{user_id}.csv"))
            similarity.to_csv(os.path.join(matrix_dir, f"ma_tran_tuong_tu_nguoi_dung_{user_id}.csv"))
            completed_norm.to_csv(os.path.join(matrix_dir, f"ma_tran_tien_ich_chuan_hoa_sau_hoan_thien_{user_id}.csv"))
            completed_util.to_csv(os.path.join(matrix_dir, f"ma_tran_tien_ich_sau_hoan_thien_{user_id}.csv"))
        except OSError as e:
            print(f"Error creating directory for user {user_id} utility matrix: {e}")

        # Extend test with random movies (excluded from eval if rating is NaN)
        extra_movies = get_random_movies_from_other_users(ratings_df, movies_df, user_id, num_movies=5, rng=rng)
        extended_test_data = pd.concat([test_data, extra_movies], ignore_index=True)

        required_columns = ["userId", "movieId", "title", "genres", "rating"]
        if not all(col in train_data.columns for col in required_columns):
            print(f"Skipping user {user_id}: Missing required columns.")
            continue

        user_dir = os.path.join("extend_train_test", f"user_{user_id}")
        try:
            os.makedirs(user_dir, exist_ok=True)
            train_data[required_columns].to_csv(os.path.join(user_dir, f"extend_train_{user_id}.csv"), index=False)
            extended_test_data[required_columns].to_csv(os.path.join(user_dir, f"extend_test_{user_id}.csv"), index=False)
        except OSError as e:
            print(f"Error creating directory for user {user_id}: {e}")

        # Predictions
        extended_test_data["predicted_rating"] = extended_test_data.apply(
            lambda row: predict_rating(
                user_id,
                int(row["movieId"]),
                similarity,
                ratings_df,
                user_means,
                k=k_neighbors,
                min_rating=min_rating,
                max_rating=max_rating,
            ),
            axis=1,
        )

        # Save predictions
        predict_dir = os.path.join("Du_doan_trong_test", f"user_{user_id}")
        try:
            os.makedirs(predict_dir, exist_ok=True)
            extended_test_data.to_csv(os.path.join(predict_dir, f"ket_qua_du_doan_{user_id}.csv"), index=False)
        except OSError as e:
            print(f"Error creating directory for user {user_id} prediction: {e}")

        # Console RMSE/MAE (optional)
        try:
            evaluate_predictions_console(extended_test_data, user_id)
        except Exception as e:
            print(f"[User {user_id}] Evaluation error: {e}")

        # Precision@K & Recall@K (K fixed)
        precision_k, recall_k = precision_recall_at_k(extended_test_data, like_threshold=like_threshold, k=k_at)

        # Per-user minimal file: only precision, recall
        eval_dir = os.path.join(eval_root, f"user_{user_id}")
        try:
            os.makedirs(eval_dir, exist_ok=True)
            pd.DataFrame([{ "precision": precision_k, "recall": recall_k }]).to_csv(
                os.path.join(eval_dir, f"evaluate_{user_id}.csv"), index=False
            )
            # Mark as processed
            append_processed_user(user_id, eval_root=eval_root)
        except OSError as e:
            print(f"Error creating directory for user {user_id} evaluate: {e}")

        # Cleanup per user
        try:
            del user_movies, train_data, test_data, train_movie_ids
            del binary, normalized, similarity, completed_norm, completed_util
            del extended_test_data, extra_movies
        except Exception:
            pass
        gc.collect()

    # Global average (resume-safe): scan disk
    recompute_global_avg(eval_root=eval_root)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--k_neighbors", type=int, default=2)
    parser.add_argument("--like_threshold", type=float, default=3.5)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--k_at", type=int, default=10)
    parser.add_argument("--start_at", type=int, default=4800, help="start index in filtered user list")
    parser.add_argument("--limit", type=int, default=300, help="process at most N users (default 300; 0=no limit)")
    parser.add_argument("--resume", action="store_true", help="skip users already evaluated/recorded")
    parser.add_argument("--force", action="store_true", help="force recompute even if evaluate exists/recorded")
    parser.add_argument("--eval_root", type=str, default="evaluate")

    # Colab/Jupyter safe: ignore unknown args like -f <kernel.json>
    args, _unknown = parser.parse_known_args()

    main(k_neighbors=args.k_neighbors,
         like_threshold=args.like_threshold,
         seed=args.seed,
         k_at=args.k_at,
         start_at=args.start_at,
         limit=args.limit,
         resume=args.resume,
         force=args.force,
         eval_root=args.eval_root)


/tmp/ipython-input-334876317.py:49: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df["movieId"] = pd.to_numeric(df["movieId"], errors="ignore")
/tmp/ipython-input-334876317.py:51: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df["userId"] = pd.to_numeric(df["userId"], errors="ignore")


[INFO] Planning to process 300 users in this run. (resume=OFF)


Processing users:   0%|          | 1/300 [00:29<2:24:34, 29.01s/it]

[User 4801] RMSE=0.7456 | MAE=0.4507


Processing users:   1%|          | 2/300 [01:07<2:51:53, 34.61s/it]

...

Processing users: 100%|█████████▉| 299/300 [2:08:56<00:27, 27.01s/it]

[User 5099] RMSE=1.0024 | MAE=0.8031


Processing users: 100%|██████████| 300/300 [2:09:40<00:00, 25.94s/it]

[User 5100] RMSE=0.9337 | MAE=0.6515


[AVG (disk)] precision=0.6807 | recall=0.5931
